# Challenge AI Engineer

## Daftar Isi

- **Soal 1B**
- **Soal 2**
- **Soal 3**

## Highlights

- Implemented a CLI-style FAQ chatbot using Ollama Cloud models, added model selection so the user can choose which cloud model to use.

## Soal 1B

In [21]:
!pip -q install ollama requests python-dotenv


In [22]:
import os
import re
from difflib import SequenceMatcher
from pathlib import Path
from getpass import getpass

import requests
from dotenv import load_dotenv
from ollama import Client

load_dotenv()
print('Library siap digunakan.')


Library siap digunakan.


### Disini nanti penjelasan ollama api key

In [23]:
OLLAMA_API_KEY = os.getenv('OLLAMA_API_KEY')
if not OLLAMA_API_KEY:
    OLLAMA_API_KEY = getpass('Masukkan OLLAMA_API_KEY dari Ollama Cloud: ').strip()

if not OLLAMA_API_KEY:
    raise ValueError('OLLAMA_API_KEY tidak boleh kosong.')

OLLAMA_API_KEY = OLLAMA_API_KEY.strip()
OLLAMA_HEADERS = {'Authorization': f'Bearer {OLLAMA_API_KEY}'}
client = Client(host='https://ollama.com', headers=OLLAMA_HEADERS)
print('API key siap.')


API key siap.


In [24]:
def fetch_cloud_models():
    response = requests.get('https://ollama.com/api/tags', headers=OLLAMA_HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()
    models = []
    for item in payload.get('models', []):
        name = item.get('name')
        if name and name not in models:
            models.append(name)
    return models


def is_subscription_error(exc):
    message = str(exc).lower()
    return ('requires a subscription' in message) or ('upgrade for access' in message) or ('status code: 403' in message)


def probe_model_access(model_name):
    try:
        client.chat(
            model=model_name,
            messages=[{'role': 'user', 'content': 'Reply with OK.'}],
            stream=False,
        )
        return True, None
    except Exception as exc:
        if is_subscription_error(exc):
            return False, 'subscription'
        return False, str(exc)


def discover_free_cloud_models():
    all_models = fetch_cloud_models()
    free_models = []
    blocked_models = []
    for model_name in all_models:
        ok, reason = probe_model_access(model_name)
        if ok:
            free_models.append(model_name)
        else:
            blocked_models.append((model_name, reason))
    return free_models, blocked_models


available_models, blocked_models = discover_free_cloud_models()
print('\nModel cloud yang bisa dipakai tanpa subscription:')
for idx, model_name in enumerate(available_models, start=1):
    print(f'{idx}. {model_name}')

if blocked_models:
    print('\nModel yang disembunyikan karena subscription / error akses:')
    for model_name, reason in blocked_models:
        print(f'- {model_name} ({reason})')

if not available_models:
    raise RuntimeError('Tidak ada model cloud yang bisa dipakai pada akun ini.')

def choose_model(prompt=None, default_model=None):
    if prompt is None:
        return default_model or available_models[0]
    choice = input(f'\n{prompt}').strip()
    if choice.startswith('/'):
        choice = choice[1:].strip()
    if choice.isdigit() and 1 <= int(choice) <= len(available_models):
        return available_models[int(choice) - 1]
    if choice:
        return choice
    return default_model or available_models[0]

active_model = choose_model(default_model=available_models[0])
print(f'Model default aktif: {active_model}')



Model cloud yang bisa dipakai tanpa subscription:
1. minimax-m3
2. minimax-m2
3. ministral-3:3b
4. devstral-small-2:24b
5. rnj-1:8b
6. nemotron-3-super
7. glm-4.6
8. ministral-3:8b
9. gemma3:4b
10. gemma3:12b
11. gpt-oss:20b
12. qwen3-coder-next
13. gpt-oss:120b
14. gemma3:27b
15. qwen3-coder:480b
16. qwen3-next:80b
17. qwen3-vl:235b-instruct
18. qwen3-vl:235b
19. minimax-m2.5
20. devstral-2:123b
21. cogito-2.1:671b
22. glm-4.7
23. nemotron-3-nano:30b
24. minimax-m2.1
25. ministral-3:14b
26. gemma4:31b
27. nemotron-3-ultra

Model yang disembunyikan karena subscription / error akses:
- kimi-k2.5 (subscription)
- glm-5.1 (subscription)
- kimi-k2-thinking (subscription)
- minimax-m2.7 (subscription)
- gemini-3-flash-preview (subscription)
- qwen3.5:397b (subscription)
- kimi-k2.6 (subscription)
- deepseek-v3.1:671b (subscription)
- glm-5 (subscription)
- deepseek-v4-flash (subscription)
- mistral-large-3:675b (subscription)
- deepseek-v3.2 (subscription)
- deepseek-v4-pro (subscription)


In [25]:
faq_entries = [
    ('Apa itu Piala Dunia FIFA 2026?', 'Piala Dunia FIFA 2026 adalah turnamen sepak bola internasional pria yang diikuti oleh tim nasional dari berbagai negara di dunia.'),
    ('Kapan Piala Dunia FIFA 2026 berlangsung?', 'Piala Dunia FIFA 2026 berlangsung dari 11 Juni 2026 sampai 19 Juli 2026.'),
    ('Negara mana saja yang menjadi tuan rumah Piala Dunia 2026?', 'Piala Dunia 2026 diselenggarakan di tiga negara, yaitu Kanada, Meksiko, dan Amerika Serikat.'),
    ('Berapa jumlah peserta Piala Dunia 2026?', 'Piala Dunia 2026 diikuti oleh 48 tim nasional.'),
    ('Berapa jumlah pertandingan di Piala Dunia 2026?', 'Piala Dunia 2026 memiliki total 104 pertandingan.'),
    ('Berapa kota yang menjadi tuan rumah Piala Dunia 2026?', 'Piala Dunia 2026 dimainkan di 16 kota tuan rumah di Kanada, Meksiko, dan Amerika Serikat.'),
    ('Apa yang membuat Piala Dunia 2026 berbeda dari edisi sebelumnya?', 'Piala Dunia 2026 menjadi edisi pertama yang diikuti oleh 48 tim dan diselenggarakan oleh tiga negara tuan rumah.'),
    ('Bagaimana format grup Piala Dunia 2026?', 'Piala Dunia 2026 menggunakan format 12 grup, dengan masing-masing grup berisi 4 tim.'),
    ('Berapa tim yang lolos dari fase grup?', 'Dua tim teratas dari setiap grup dan delapan tim peringkat ketiga terbaik lolos ke babak gugur.'),
    ('Berapa tim yang bermain di babak gugur?', 'Sebanyak 32 tim bermain di babak gugur.'),
    ('Babak apa saja yang ada di fase gugur Piala Dunia 2026?', 'Fase gugur terdiri dari babak 32 besar, 16 besar, perempat final, semifinal, perebutan tempat ketiga, dan final.'),
    ('Di mana pertandingan pembuka Piala Dunia 2026 dimainkan?', 'Pertandingan pembuka dimainkan di Mexico City Stadium, Meksiko, pada 11 Juni 2026.'),
    ('Siapa yang bermain pada pertandingan pembuka Piala Dunia 2026?', 'Pertandingan pembuka mempertemukan Meksiko melawan Afrika Selatan.'),
    ('Di mana final Piala Dunia 2026 dimainkan?', 'Final Piala Dunia 2026 dimainkan di New York New Jersey Stadium.'),
    ('Apakah Indonesia menjadi tuan rumah Piala Dunia 2026?', 'Tidak. Indonesia bukan tuan rumah Piala Dunia 2026. Tuan rumahnya adalah Kanada, Meksiko, dan Amerika Serikat.'),
    ('Apakah Indonesia ikut Piala Dunia 2026?', 'Informasi ini bergantung pada hasil kualifikasi resmi. Chatbot ini hanya menjawab berdasarkan FAQ yang tersedia.'),
    ('Di mana penonton Indonesia bisa melihat jadwal resmi Piala Dunia 2026?', 'Penonton Indonesia dapat melihat jadwal resmi melalui situs resmi FIFA.'),
    ('Apakah jadwal pertandingan Piala Dunia 2026 menggunakan waktu Indonesia?', 'Jadwal resmi FIFA biasanya menggunakan waktu lokal venue. Penonton Indonesia perlu mengonversi jadwal tersebut ke WIB, WITA, atau WIT.'),
    ('Apa perbedaan WIB, WITA, dan WIT?', 'WIB adalah Waktu Indonesia Barat, WITA adalah Waktu Indonesia Tengah, dan WIT adalah Waktu Indonesia Timur.'),
    ('Apakah pertandingan Piala Dunia 2026 bisa ditonton dari Indonesia?', 'Pertandingan dapat ditonton dari Indonesia jika tersedia melalui pemegang hak siar resmi atau platform legal yang menayangkan Piala Dunia 2026.'),
    ('Di mana membeli tiket resmi Piala Dunia 2026?', 'Tiket resmi Piala Dunia 2026 hanya boleh dibeli melalui kanal resmi FIFA.'),
    ('Apakah aman membeli tiket dari pihak ketiga?', 'Penonton sebaiknya berhati-hati terhadap tiket dari pihak ketiga dan memprioritaskan pembelian melalui kanal resmi FIFA.'),
    ('Apakah penonton Indonesia perlu visa untuk menonton langsung Piala Dunia 2026?', 'Jika ingin menonton langsung di Kanada, Meksiko, atau Amerika Serikat, penonton Indonesia perlu memeriksa persyaratan visa sesuai negara tujuan.'),
    ('Apakah satu visa bisa dipakai untuk semua negara tuan rumah?', 'Tidak selalu. Kanada, Meksiko, dan Amerika Serikat memiliki aturan visa masing-masing. Penonton harus memeriksa aturan resmi negara yang akan dikunjungi.'),
    ('Apakah chatbot ini bisa memberi prediksi pemenang Piala Dunia 2026?', 'Tidak. Chatbot ini hanya menjawab berdasarkan FAQ dan tidak memberikan prediksi.'),
    ('Apakah chatbot ini bisa memberi skor langsung?', 'Tidak. Chatbot ini tidak menyediakan live score. Untuk skor langsung, pengguna perlu melihat situs resmi FIFA atau sumber live score terpercaya.'),
    ('Apakah chatbot ini bisa menjawab pertanyaan tentang pemain tertentu?', 'Hanya jika informasi pemain tersebut tersedia di FAQ. Jika tidak ada, chatbot harus menolak menjawab.'),
    ('Apa yang harus dijawab chatbot jika pertanyaan tidak ada di FAQ?', 'Maaf, saya tidak dapat membantu dengan pertanyaan itu karena informasi tersebut tidak tersedia di FAQ.'),
    ('Apa yang harus dijawab chatbot jika pengguna bertanya hal di luar Piala Dunia 2026?', 'Maaf, saya hanya dapat menjawab pertanyaan berdasarkan FAQ Piala Dunia 2026 untuk penonton Indonesia.'),
    ('Apa contoh pertanyaan yang bisa dijawab chatbot ini?', 'Chatbot ini bisa menjawab pertanyaan tentang jadwal umum, negara tuan rumah, jumlah tim, format turnamen, tiket resmi, dan informasi dasar untuk penonton Indonesia.'),
]

faq_path = Path('faqs.txt')
faq_path.write_text('\n'.join(f'Q: {q}\nA: {a}\n' for q, a in faq_entries), encoding='utf-8')
print(f'faqs.txt dibuat di: {faq_path.resolve()}')
print(f'Jumlah FAQ: {len(faq_entries)}')


faqs.txt dibuat di: /content/faqs.txt
Jumlah FAQ: 30


In [26]:
def load_faq(path='faqs.txt'):
    lines = Path(path).read_text(encoding='utf-8').splitlines()
    pairs = []
    current_q = None
    current_a = None
    for line in lines:
        line = line.strip()
        if line.startswith('Q:'):
            current_q = line[2:].strip()
        elif line.startswith('A:'):
            current_a = line[2:].strip()
        if current_q and current_a:
            pairs.append((current_q, current_a))
            current_q = None
            current_a = None
    return pairs


def normalize(text):
    return re.findall(r'[a-z0-9]+', text.lower())


def similarity(a, b):
    tokens_a = set(normalize(a))
    tokens_b = set(normalize(b))
    if not tokens_a or not tokens_b:
        return 0.0
    jaccard = len(tokens_a & tokens_b) / len(tokens_a | tokens_b)
    ratio = SequenceMatcher(None, a.lower(), b.lower()).ratio()
    return (0.7 * jaccard) + (0.3 * ratio)


def best_faq_match(question, faq_pairs):
    scored = []
    for faq_question, faq_answer in faq_pairs:
        scored.append((similarity(question, faq_question), faq_question, faq_answer))
    scored.sort(reverse=True, key=lambda item: item[0])
    return scored[0] if scored else (0.0, '', '')


faq_pairs = load_faq()
print(f'FAQ loaded: {len(faq_pairs)} item')
for idx, (q, a) in enumerate(faq_pairs[:5], start=1):
    print(f'{idx}. {q} -> {a}')


FAQ loaded: 30 item
1. Apa itu Piala Dunia FIFA 2026? -> Piala Dunia FIFA 2026 adalah turnamen sepak bola internasional pria yang diikuti oleh tim nasional dari berbagai negara di dunia.
2. Kapan Piala Dunia FIFA 2026 berlangsung? -> Piala Dunia FIFA 2026 berlangsung dari 11 Juni 2026 sampai 19 Juli 2026.
3. Negara mana saja yang menjadi tuan rumah Piala Dunia 2026? -> Piala Dunia 2026 diselenggarakan di tiga negara, yaitu Kanada, Meksiko, dan Amerika Serikat.
4. Berapa jumlah peserta Piala Dunia 2026? -> Piala Dunia 2026 diikuti oleh 48 tim nasional.
5. Berapa jumlah pertandingan di Piala Dunia 2026? -> Piala Dunia 2026 memiliki total 104 pertandingan.


In [27]:
FALLBACK_ANSWER = 'Maaf, saya tidak dapat membantu dengan pertanyaan itu.'
MAX_CONTEXT_MESSAGES = 8

SYSTEM_PROMPT = (
    'Anda adalah chatbot FAQ Piala Dunia FIFA 2026. '
    'Jawab hanya berdasarkan konteks FAQ yang diberikan dan riwayat percakapan yang relevan. '
    'Jangan menambahkan fakta baru, jangan berasumsi, dan jangan menjawab di luar konteks. '
    f'Jika konteks tidak cukup, jawab persis: {FALLBACK_ANSWER}'
)

conversation = []

def build_messages(user_question, matched_question, matched_answer):
    faq_context = (
        'Konteks FAQ yang relevan:\n'
        f'Pertanyaan FAQ: {matched_question}\n'
        f'Jawaban FAQ: {matched_answer}\n\n'
        f'Pertanyaan pengguna: {user_question}\n'
        'Jawab singkat dan hanya berdasarkan jawaban FAQ di atas.'
    )
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT + '\n\n' + faq_context}]
    messages.extend(conversation[-MAX_CONTEXT_MESSAGES:])
    messages.append({'role': 'user', 'content': user_question})
    return messages


def ask_model(user_question):
    score, matched_question, matched_answer = best_faq_match(user_question, faq_pairs)
    if score < 0.20:
        return FALLBACK_ANSWER

    messages = build_messages(user_question, matched_question, matched_answer)
    try:
        response = client.chat(
            model=active_model,
            messages=messages,
            stream=False,
        )
        text = response.get('message', {}).get('content', '').strip()
        return text or matched_answer
    except Exception as exc:
        if is_subscription_error(exc):
            print(f'\n[Model {active_model} tidak tersedia karena subscription. Silakan ganti model.]')
            return FALLBACK_ANSWER
        print(f'\n[Fallback karena error Ollama Cloud: {exc}]')
        return matched_answer


def show_history():
    if not conversation:
        print('Belum ada riwayat percakapan.')
        return
    print('\nRiwayat konteks percakapan terakhir:')
    for item in conversation[-MAX_CONTEXT_MESSAGES:]:
        print(f"- {item['role']}: {item['content']}")


def switch_model():
    global active_model
    print('\nModel yang tersedia:')
    for idx, model_name in enumerate(available_models, start=1):
        print(f'{idx}. {model_name}')
    new_model = choose_model('Pilih model baru dengan nomor, atau ketik nama model langsung: ')
    active_model = new_model
    print(f'Model aktif sekarang: {active_model}')


def run_chatbot():
    print('\nChatbot FAQ siap digunakan.')
    print(f'Model aktif: {active_model}')
    print('Ketik /model, /models, /history, /reset, exit, atau quit.')

    while True:
        user_input = input('\nUser: ').strip()
        if not user_input:
            print('Bot: Silakan masukkan pertanyaan.')
            continue

        lowered = user_input.lower()
        if lowered in {'exit', 'quit'}:
            print('Bot: Terima kasih. Sesi chatbot selesai.')
            break
        if lowered == '/models':
            print('\nModel yang bisa dipakai:')
            for idx, model_name in enumerate(available_models, start=1):
                print(f'{idx}. {model_name}')
            continue
        if lowered == '/model':
            switch_model()
            continue
        if lowered == '/history':
            show_history()
            continue
        if lowered == '/reset':
            conversation.clear()
            print('Bot: Riwayat percakapan telah dihapus.')
            continue

        print('Bot: ', end='', flush=True)
        response = ask_model(user_input)
        print(response)
        conversation.append({'role': 'user', 'content': user_input})
        conversation.append({'role': 'assistant', 'content': response})


run_chatbot()



Chatbot FAQ siap digunakan.
Model aktif: minimax-m3
Ketik /model, /models, /history, /reset, exit, atau quit.

User: halo
Bot: Maaf, saya tidak dapat membantu dengan pertanyaan itu.

User: indo ikut ga
Bot: Informasi ini bergantung pada hasil kualifikasi resmi. Chatbot ini hanya menjawab berdasarkan FAQ yang tersedia.

User: exit
Bot: Terima kasih. Sesi chatbot selesai.


## Bagian 2 - Teori AI

### Soal 2: Pertanyaan Teori Dasar

**a. Apa yang dimaksud dengan Artificial Intelligence (AI)? Sebutkan dua contohnya dalam kehidupan sehari-hari.**

Artificial Intelligence (AI) adalah bidang ilmu komputer yang membuat mesin mampu meniru kemampuan cerdas manusia, seperti mengenali pola, memahami bahasa, mengambil keputusan, dan belajar dari data.

Dua contoh AI dalam kehidupan sehari-hari:

- Rekomendasi video di YouTube atau Netflix.
- Asisten virtual seperti Siri, Google Assistant, atau ChatGPT.

**b. Apa perbedaan antara Supervised Learning dan Unsupervised Learning? Berikan satu contoh untuk masing-masing.**

- **Supervised Learning** menggunakan data yang sudah memiliki label jawaban. Model belajar dari pasangan input-output yang benar. Contoh: klasifikasi email spam dan bukan spam.
- **Unsupervised Learning** menggunakan data tanpa label. Model mencari pola atau struktur sendiri. Contoh: clustering pelanggan berdasarkan perilaku belanja.

### Soal 3: Pertanyaan Konsep

**a. Apa itu Feature dalam konteks machine learning? Mengapa penting untuk memilih fitur yang tepat saat membangun model?**

Feature adalah variabel atau atribut yang digunakan model sebagai masukan untuk mempelajari pola. Contohnya umur, pendapatan, atau jumlah klik.

Pemilihan fitur yang tepat penting karena fitur yang relevan membantu model belajar lebih akurat, lebih cepat, dan lebih stabil. Fitur yang kurang tepat dapat membuat model sulit belajar atau menghasilkan prediksi yang kurang baik.

**b. Apa itu Fine-tuning dalam machine learning? Sebutkan satu kasus di mana fine-tuning berguna.**

Fine-tuning adalah proses menyesuaikan model yang sudah pre-trained agar lebih cocok dengan tugas atau data yang lebih spesifik.

Contoh kasus yang berguna: model bahasa umum di-fine-tune untuk klasifikasi sentimen ulasan pelanggan pada domain e-commerce atau layanan keuangan.
